# Semantic Drone Segmentation Pipeline

This notebook trains **two** segmentation architectures on the [Semantic Drone Dataset](https://www.tugraz.at/index.php?id=22387) (400 aerial images, 23 classes):

1. **Improved U-Net** — BatchNorm, augmentation, train/val split, Mean IoU tracking.
2. **DeepLabV3+** (ResNet50 backbone, ImageNet pretrained) — two-phase training (warm-up → fine-tune) with Atrous Spatial Pyramid Pooling for multi-scale context.

All code lives in the `src/` package.

## 1. Setup

In [ ]:
import os

# ── Data paths (update if your extraction layout differs) ──
IMAGES_DIR = 'data/original_images/'
MASKS_DIR  = 'data/label_images/'

UNET_OUTPUT_DIR   = 'data/output/unet/'
DEEPLABV3_OUTPUT_DIR = 'data/output/deeplabv3plus/'

os.makedirs(IMAGES_DIR, exist_ok=True)
os.makedirs(MASKS_DIR, exist_ok=True)

## 2. Train the Improved U-Net

Includes BatchNormalization, data augmentation, 80/20 train/val split, Mean IoU, and `ReduceLROnPlateau`.

In [ ]:
from src.train import train_unet

unet_model, unet_history = train_unet(
    images_dir=IMAGES_DIR,
    masks_dir=MASKS_DIR,
    epochs=150,
    batch_size=8,
    target_size=(256, 256),
    n_classes=23,
    validation_split=0.2,
    output_dir=UNET_OUTPUT_DIR,
)

## 3. Evaluate U-Net

In [ ]:
from src.dataset import build_dataset
from src.predict import evaluate_model, visualize_predictions

# Rebuild val set for evaluation
_, val_ds = build_dataset(
    IMAGES_DIR, MASKS_DIR,
    batch_size=8, target_size=(256, 256), validation_split=0.2,
)

print("=== U-Net Evaluation ===")
unet_metrics = evaluate_model(unet_model, val_ds, n_classes=23)
visualize_predictions(unet_model, val_ds, num=4)

## 4. Train DeepLabV3+ (Advanced Model)

Uses a **ResNet50** backbone pretrained on ImageNet with **Atrous Spatial Pyramid Pooling (ASPP)** for multi-scale feature extraction. Training is done in two phases:
- **Phase 1 (warm-up)**: Backbone frozen, decoder learns from scratch at LR=1e-3.
- **Phase 2 (fine-tune)**: Entire model unfrozen, trained end-to-end at LR=1e-4.

In [ ]:
from src.train_deeplabv3plus import train_deeplabv3plus

dlv3_model, hist_warmup, hist_finetune = train_deeplabv3plus(
    images_dir=IMAGES_DIR,
    masks_dir=MASKS_DIR,
    epochs_warmup=20,
    epochs_finetune=80,
    batch_size=8,
    target_size=(256, 256),
    n_classes=23,
    validation_split=0.2,
    output_dir=DEEPLABV3_OUTPUT_DIR,
)

## 5. Evaluate DeepLabV3+ and Compare

In [ ]:
print("=== DeepLabV3+ Evaluation ===")
dlv3_metrics = evaluate_model(dlv3_model, val_ds, n_classes=23)
visualize_predictions(dlv3_model, val_ds, num=4)

In [ ]:
import matplotlib.pyplot as plt

# ── Side-by-side Mean IoU comparison ──
models = ['U-Net', 'DeepLabV3+']
ious = [unet_metrics['mean_iou'], dlv3_metrics['mean_iou']]

fig, ax = plt.subplots(figsize=(6, 4))
bars = ax.bar(models, ious, color=['#4C72B0', '#DD8452'], width=0.5)
for bar, val in zip(bars, ious):
    ax.text(bar.get_x() + bar.get_width() / 2, bar.get_height() + 0.005,
            f'{val:.4f}', ha='center', va='bottom', fontsize=12)
ax.set_ylabel('Mean IoU')
ax.set_title('Model Comparison — Validation Mean IoU')
ax.set_ylim(0, max(ious) * 1.2)
plt.tight_layout()
plt.show()

# ── Training curves ──
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Loss
axes[0].plot(unet_history.history['loss'], label='U-Net train')
axes[0].plot(unet_history.history['val_loss'], label='U-Net val', linestyle='--')
ft_loss = hist_finetune.history['loss']
ft_vloss = hist_finetune.history['val_loss']
axes[0].plot(ft_loss, label='DLv3+ train')
axes[0].plot(ft_vloss, label='DLv3+ val', linestyle='--')
axes[0].set_title('Loss')
axes[0].set_xlabel('Epoch')
axes[0].legend()

# Accuracy
axes[1].plot(unet_history.history['accuracy'], label='U-Net train')
axes[1].plot(unet_history.history['val_accuracy'], label='U-Net val', linestyle='--')
axes[1].plot(hist_finetune.history['accuracy'], label='DLv3+ train')
axes[1].plot(hist_finetune.history['val_accuracy'], label='DLv3+ val', linestyle='--')
axes[1].set_title('Pixel Accuracy')
axes[1].set_xlabel('Epoch')
axes[1].legend()

plt.tight_layout()
plt.show()